# Stochastic Calculus from Zero · Episode 17: SDEs: geometric Brownian motion solved

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The share** (Series 3's): priced at 100, 20% volatility, a 3.60% risk-free rate (continuous), a call with strike 105 expiring in a year, and an illustrative real-world drift of 8%. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. What an SDE is · 3. Guess and check · 4. One path, two ways · 5. Mean and median · 6. When a solution exists · 7. This episode's question · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 17,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. What an SDE is

A **stochastic differential equation** is a recipe for each small step of a process:

$$dX_t = \mu(t, X_t)\,dt + \sigma(t, X_t)\,dW_t .$$

The first term is the **drift**, the second the **diffusion** term (noise scaled by $\sigma$). It is shorthand for the integral equation

$$X_t = X_0 + \int_0^t \mu(s, X_s)\,ds + \int_0^t \sigma(s, X_s)\,dW_s ,$$

with the Itô integral of Part 2. A **strong solution** is a process, built from the given Brownian motion, that satisfies it. Our share follows geometric Brownian motion: $dS = \mu S\,dt + \sigma S\,dW$.

In [ ]:
S0, VOL, MU = 100.0, 0.20, 0.08
out["inputs"] = {"S0": S0, "vol": VOL, "mu": MU, "vol_pct": VOL * 100, "mu_pct": MU * 100}

## 3. Guess and check

Part 2 (Episode 15) showed that $\ln S$ drifts at $\mu - \tfrac12\sigma^2$. So guess $S_t = S_0 e^{Y_t}$ with $Y_t = (\mu - \tfrac12\sigma^2)t + \sigma W_t$. Itô-Doeblin with $f(y) = S_0e^y$ ($f' = f'' = f$) and $(dY)^2 = \sigma^2\,dt$:

$$dS = S\,dY + \tfrac12 S\,(dY)^2 = S\big((\mu - \tfrac12\sigma^2)\,dt + \sigma\,dW\big) + \tfrac12\sigma^2 S\,dt = \mu S\,dt + \sigma S\,dW. \checkmark$$

The $-\tfrac12\sigma^2$ in the guess exactly cancels the Itô term.

## 4. One path, two ways

On Episode 1's path, compare the formula with simply stepping the SDE forward (the **Euler** scheme, Episode 20): $S_{n+1} = S_n(1 + \mu\,\Delta t + \sigma\,\Delta W_n)$, 1,000 steps.

In [ ]:
rng = np.random.default_rng(seed)
t, W, dW = brownian_paths(1, 1_000, 1.0, rng)              # Episode 1's path comes first from this seed
w, dw = W[0], dW[0]
exact = S0 * np.exp((MU - 0.5 * VOL ** 2) * t + VOL * w)
euler = np.empty_like(exact); euler[0] = S0
for n, d in enumerate(dw):
    euler[n + 1] = euler[n] * (1 + MU * (t[1] - t[0]) + VOL * d)
every = 4
out["path"] = {"t": t[::every].round(4).tolist(), "exact": [exact[::every].round(3).tolist()],
               "euler": euler[::every].round(3).tolist(), "steps": len(dw), "W_T": float(w[-1]),
               "exact_T": float(exact[-1]), "euler_T": float(euler[-1]), "gap": float(euler[-1] - exact[-1]),
               "max_gap": float(np.abs(euler - exact).max()),
               "y_range": [math.floor(min(exact.min(), euler.min()) / 10) * 10, math.ceil(max(exact.max(), euler.max()) / 10) * 10],
               "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}
assert abs(out["path"]["gap"]) < 0.5
print(f"exact {exact[-1]:.2f}, Euler {euler[-1]:.2f}, largest gap along the path {out['path']['max_gap']:.2f}")

## 5. Mean and median

$\ln S_T$ is normal with mean $\ln S_0 + (\mu - \tfrac12\sigma^2)T$ and variance $\sigma^2T$, so $S_T$ is **lognormal**:

- mean $E[S_T] = S_0e^{\mu T}$ (the drift of the price),
- median $S_0e^{(\mu - \frac12\sigma^2)T}$ (the drift of the log).

The mean sits above the median: a few big winners pull the average up.

In [ ]:
ends = S0 * np.exp((MU - 0.5 * VOL ** 2) + VOL * rng.standard_normal(200_000))
_, Wf, _ = brownian_paths(40, 252, 1.0, rng)
tf = np.linspace(0, 1, 253)
fan = S0 * np.exp((MU - 0.5 * VOL ** 2) * tf + VOL * Wf)
bins = np.arange(50, 200.01, 5)
counts, _ = np.histogram(np.clip(ends, 50, 199.99), bins=bins)
mm = {"paths": len(ends), "mean": float(ends.mean()), "mean_theory": S0 * math.exp(MU),
      "median": float(np.median(ends)), "median_theory": S0 * math.exp(MU - 0.5 * VOL ** 2),
      "share_above_mean": float((ends > ends.mean()).mean()),
      "t": tf[::2].round(4).tolist(), "fan": fan[:, ::2].round(2).tolist(),
      "hist": {"y": ((bins[:-1] + bins[1:]) / 2).tolist(), "share": (counts / len(ends)).round(5).tolist()}}
assert abs(mm["mean"] / mm["mean_theory"] - 1) < 0.002 and abs(mm["median"] / mm["median_theory"] - 1) < 0.003
out["mm"] = mm
pd.DataFrame([{k: mm[k] for k in ("mean", "mean_theory", "median", "median_theory", "share_above_mean")}])

## 6. When a solution exists

If the drift and volatility are **Lipschitz** in $x$ (they change by at most a constant times the change in $x$) and grow at most linearly, the SDE has exactly one strong solution. GBM's $\mu x$ and $\sigma x$ qualify. The square root in Episode 19's CIR model does not (its slope is infinite at 0), and needs a separate argument.

## 7. This episode's question

With $\sigma = 40\%$ and $\mu = 8\%$, the median of $S_1$ is $100\,e^{0.08 - 0.08} = 100$: half the paths end below where they started, even though the mean grows 8%.

In [ ]:
out["question"] = {"vol": 0.40, "median": S0 * math.exp(MU - 0.5 * 0.40 ** 2)}
assert abs(out["question"]["median"] - 100) < 1e-9

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Solve the arithmetic SDE $dX = \mu\,dt + \sigma\,dW$, $X_0 = x$. What are the mean and variance of $X_t$?

<details><summary>Answer</summary>

Integrate both sides: $X_t = x + \mu t + \sigma W_t$. Mean $x + \mu t$, variance $\sigma^2 t$. (This is Bachelier's model.)

</details>

**Exercise 2 (By hand).** Show that $S_t = S_0\exp(\mu t + \sigma W_t)$ (no $-\tfrac12\sigma^2$) solves a different SDE. Which one?

<details><summary>Answer</summary>

Itô with $Y = \mu t + \sigma W$: $dS = S\,dY + \tfrac12 S\sigma^2\,dt = (\mu + \tfrac12\sigma^2)S\,dt + \sigma S\,dW$. Its drift is $\mu + \tfrac12\sigma^2$, not $\mu$.

</details>

**Exercise 3 (By hand).** Why is the median of $S_T$ equal to $S_0e^{(\mu - \frac12\sigma^2)T}$?

<details><summary>Answer</summary>

$S_T$ is an increasing function of $\ln S_T$, so its median is $e^{\text{median of } \ln S_T}$. $\ln S_T$ is normal, so its median is its mean, $\ln S_0 + (\mu - \tfrac12\sigma^2)T$.

</details>

**Exercise 4 (In code).** Step the SDE with 10, 100 and 1,000 steps along Episode 1's path (add up its steps in blocks). How does the gap to the exact solution change?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
gaps = {}
for n in (10, 100, 1_000):
    d = dw.reshape(n, -1).sum(axis=1)
    s = S0
    for x in d:
        s *= 1 + MU / n + VOL * x
    gaps[n] = s - exact[-1]
assert abs(gaps[1_000]) < abs(gaps[10])
print({n: round(float(g), 3) for n, g in gaps.items()}, "(fewer steps, bigger gap)")

**Exercise 5 (In code).** What is the real-world chance that the share ends the year above 100? Compute $N\big((\mu - \tfrac12\sigma^2)/\sigma\big)$ and check it on the 200,000 simulated ends.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
p = float(norm_cdf((MU - 0.5 * VOL ** 2) / VOL))
sim = float((ends > S0).mean())
assert abs(p - sim) < 0.005
print(f"formula {p:.4f}, simulated {sim:.4f}")

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")